In [1]:
from ingestion import get_transcripts_dataframe, build_index
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
import json
import pandas as pd

openai_client = OpenAI()

In [2]:
documents_df=get_transcripts_dataframe()
index = build_index(documents_df)

In [3]:
documents_list = []

for doc in documents_df:
    documents_list.append(doc)

In [4]:
data_gen_instructions = """
You emulate a user who's want to know more about self-awareness.
Formulate 5 questions this user might ask based on a dataset. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()



In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
from evaluation_utils import llm_structured

In [7]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [8]:
doc = documents_list[1]

In [9]:
documents_list[1]

{'id': 1,
 'question': 'What are the four main domains of emotional intelligence?',
 'answer': 'The four domains are self-awareness, self-management, empathy, and relationship management.',
 'video_id': 'BqF50IuR3_c'}

In [10]:
generate_ground_truth(doc)

([{'question': 'What are the four main parts of emotional intelligence?',
   'document': 1},
  {'question': 'Which four domains make up emotional intelligence?',
   'document': 1},
  {'question': 'Can you list the four key areas of emotional intelligence?',
   'document': 1},
  {'question': 'What are the main emotional intelligence domains people talk about?',
   'document': 1},
  {'question': 'What are the four pillars of emotional intelligence?',
   'document': 1}],
 ResponseUsage(input_tokens=188, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=65, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=253))

In [11]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents_list[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [12]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [13]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents_list, generate_ground_truth)

  0%|          | 0/100 [00:00<?, ?it/s]

In [14]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

## Total Cost

In [15]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.05595825000000001

In [16]:
df_ground_truth = pd.DataFrame(ground_truth)

In [17]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [18]:
len(df_ground_truth)

500